In [ ]:
import xarray as xr
import numpy as np
import zarr
import dask
from pathlib import Path
import json
from datetime import datetime

class WeatherBench2Downloader:
    def __init__(self, output_dir="./weatherbench2_data"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # WeatherBench2 ERA5 dataset URLs
        self.wb2_urls = {
            '0.25deg_6hourly': "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-0p25deg-chunk-1.zarr-v2",
            '1.40625deg_6hourly': "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-1p40625deg-chunk-1.zarr-v2"
        }
        
        # Europe bounds for subsetting
        self.europe_bounds = {
            'latitude': slice(75, 30),    # North to South
            'longitude': slice(-25, 50)   # West to East (includes Atlantic)
        }
        
    def download_era5_for_mesa(self, 
                              resolution='0.25deg_6hourly',
                              years=slice('2010', '2023'),
                              save_local=True):
        """Download WeatherBench2 ERA5 optimized for MESA-Net"""
        
        print("WeatherBench2 ERA5 Download for MESA-Net")
        print("=" * 50)
        print(f"Resolution: {resolution}")
        print(f"Years: {years}")
        print(f"Region: Europe {self.europe_bounds}")
        
        try:
            # Open the WeatherBench2 dataset
            print("\n1. Connecting to WeatherBench2...")
            ds_full = xr.open_zarr(
                self.wb2_urls[resolution],
                chunks={'time': 100}  # Chunk for memory efficiency
            )
            
            print(f"✓ Connected successfully")
            print(f"  Full dataset shape: {dict(ds_full.dims)}")
            print(f"  Available variables: {list(ds_full.data_vars.keys())}")
            print(f"  Time range: {ds_full.time.values[0]} to {ds_full.time.values[-1]}")
            
            # Select time period
            print(f"\n2. Selecting time period: {years}")
            ds_time = ds_full.sel(time=years)
            
            # Select Europe region
            print(f"3. Selecting Europe region...")
            ds_europe = ds_time.sel(**self.europe_bounds)
            
            print(f"✓ Europe subset created")
            print(f"  Europe shape: {dict(ds_europe.dims)}")
            print(f"  Spatial coverage: {ds_europe.latitude.values.min():.1f}°N to {ds_europe.latitude.values.max():.1f}°N")
            print(f"                    {ds_europe.longitude.values.min():.1f}°E to {ds_europe.longitude.values.max():.1f}°E")
            
            # Select variables relevant for precipitation forecasting
            precip_variables = self.get_precipitation_variables(ds_europe)
            ds_selected = ds_europe[precip_variables]
            
            print(f"\n4. Selected variables for precipitation forecasting:")
            for i, var in enumerate(precip_variables, 1):
                print(f"  {i:2d}. {var}")
            
            # Save locally if requested
            if save_local:
                output_file = self.output_dir / f"era5_mesa_{resolution}_{years.start}_{years.stop}.zarr"
                print(f"\n5. Saving to local storage: {output_file}")
                
                # Save with optimized chunking for MESA-Net
                ds_selected.chunk({
                    'time': 100,
                    'latitude': -1, 
                    'longitude': -1
                }).to_zarr(output_file, mode='w')
                
                print(f"✓ Dataset saved locally")
                
                # Create metadata file
                self.save_metadata(ds_selected, output_file)
                
                return output_file, ds_selected
            
            return None, ds_selected
            
        except Exception as e:
            print(f"✗ Error downloading WeatherBench2 data: {e}")
            print("\nTrying alternative access method...")
            return self.download_via_http(resolution, years)
    
    def get_precipitation_variables(self, dataset):
        """Select variables most relevant for precipitation forecasting"""
        
        # Check what's actually available in the dataset
        available_vars = list(dataset.data_vars.keys())
        
        # Priority order for precipitation forecasting
        desired_vars = [
            # Core precipitation and moisture
            'total_precipitation_6hr',        # Target variable
            'total_precipitation_hourly',     # Alternative precipitation
            'total_column_water_vapour',      # Atmospheric moisture
            
            # Temperature
            '2m_temperature',                 # Surface temperature
            'temperature',                    # Multi-level temperature
            
            # Pressure and dynamics
            'mean_sea_level_pressure',        # Sea level pressure
            'surface_pressure',               # Surface pressure
            'geopotential',                   # Height fields
            
            # Wind
            '10m_u_component_of_wind',        # Surface wind U
            '10m_v_component_of_wind',        # Surface wind V
            'u_component_of_wind',            # Multi-level wind U
            'v_component_of_wind',            # Multi-level wind V
            
            # Moisture and clouds
            'specific_humidity',              # Specific humidity
            'relative_humidity',              # Relative humidity
            'total_cloud_cover',              # Cloud cover
            
            # Vertical motion
            'vertical_velocity',              # Omega (vertical motion)
        ]
        
        # Select only variables that exist in the dataset
        selected_vars = []
        for var in desired_vars:
            if var in available_vars:
                selected_vars.append(var)
        
        print(f"Selected {len(selected_vars)} out of {len(available_vars)} available variables")
        print(f"Selected vars are: " + ", ".join(selected_vars))
        
        return selected_vars
    
    def download_via_http(self, resolution, years):
        """Alternative download method if GCS access fails"""
        
        print("Attempting HTTP access to WeatherBench2...")
        
        # HTTP URLs (if available)
        http_urls = {
            '0.25deg_6hourly': "https://storage.googleapis.com/weatherbench2/datasets/era5/1959-2023_01_10-6h-0p25deg-chunk-1.zarr-v2"
        }
        
        if resolution in http_urls:
            try:
                ds = xr.open_zarr(http_urls[resolution])
                # Continue with same processing...
                return self.process_dataset(ds, years)
            except Exception as e:
                print(f"HTTP access also failed: {e}")
                return None, None
        else:
            print("HTTP URLs not available for this resolution")
            return None, None
    
    def save_metadata(self, dataset, output_file):
        """Save dataset metadata for future reference"""
        
        metadata = {
            'source': 'WeatherBench2 ERA5',
            'download_date': datetime.now().isoformat(),
            'dataset_shape': dict(dataset.dims),
            'variables': list(dataset.data_vars.keys()),
            'time_range': {
                'start': str(dataset.time.values[0]),
                'end': str(dataset.time.values[-1]),
                'frequency': '6-hourly'
            },
            'spatial_coverage': {
                'lat_min': float(dataset.latitude.min().values),
                'lat_max': float(dataset.latitude.max().values),
                'lon_min': float(dataset.longitude.min().values),
                'lon_max': float(dataset.longitude.max().values),
                'resolution': '0.25 degrees'
            },
            'file_path': str(output_file),
            'preprocessing': 'WeatherBench2 standard preprocessing applied'
        }
        
        metadata_file = output_file.parent / f"{output_file.stem}_metadata.json"
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ Metadata saved: {metadata_file}")
        
        return metadata

# Quick download function
def quick_download_for_mesa():
    """Quick download setup for MESA-Net development"""
    
    downloader = WeatherBench2Downloader()
    
    # Download recent years for faster development
    output_file, dataset = downloader.download_era5_for_mesa(
        resolution='0.25deg_6hourly',
        years=slice('2015', '2023'),  # 8 years for development
        save_local=True
    )
    
    if output_file and dataset:
        print(f"\n{'='*50}")
        print("DOWNLOAD SUCCESSFUL!")
        print(f"{'='*50}")
        print(f"Local file: {output_file}")
        print(f"Dataset shape: {dict(dataset.dims)}")
        print(f"Variables: {len(dataset.data_vars)}")
        print(f"Ready for MESA-Net development!")
        
        return output_file, dataset
    else:
        print("Download failed. Check your internet connection and try again.")
        return None, None

# Alternative: Full historical download
def full_download_for_mesa():
    """Full historical download for comprehensive training"""
    
    downloader = WeatherBench2Downloader()
    
    # Download full available period
    output_file, dataset = downloader.download_era5_for_mesa(
        resolution='0.25deg_6hourly',
        years=slice('1979', '2023'),  # Full period
        save_local=True
    )
    
    return output_file, dataset

# ============ MAIN EXECUTION ============

if __name__ == "__main__":
    print("WeatherBench2 ERA5 Downloader for MESA-Net")
    print("Choose your download strategy:")
    print("1. Quick download (2015-2023, ~8 years)")
    print("2. Full download (1979-2023, ~44 years)")
    
    # Recommended: Start with quick download
    print("\nStarting quick download for development...")
    output_file, dataset = full_download_for_mesa()
    
    # Uncomment for full download:
    # print("\nStarting full historical download...")
    # output_file, dataset = full_download_for_mesa()